# FastText — kaise run karein (13 Oct)

**2 parts hain:**
1. **Pretrained model** — dataset/file `wiki-news-300d-1M.vec` chahiye (yeh repo me NAHI hai, neeche se download karo)
2. **Train your own FastText** — koi dataset NAHI chahiye, 10 sentences notebook me hi hardcoded hain

> **Dataset `.txt/.vec` me hi hota hai — ye normal hai!** `.vec` file ek plain-text file hai:
> pehli line `1000000 300` (words, dimensions), uske baad har line = `word 0.12 -0.03 ...` (300 numbers).
> Isliye code me `binary=False` likha hai — gensim `.txt/.vec` ko directly padh leta hai, convert karne ki zaroorat nahi.

## Step 0 — Setup (sirf pehli baar)
Terminal / Anaconda Prompt me:
```bash
pip install gensim jupyter
jupyter notebook FastText.ipynb
```
Phir har cell ko upar se neeche `Shift + Enter` se run karo.

## Step 1 — Pretrained dataset download karo (Part 1 ke liye)
Option A (original, Facebook): https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip  (~1.1 GB zip, extract karne par ~5 GB `.vec`)  
Option B (Kaggle): `wiki-news-300d-1M.vec` search karke download karo

**Extract karke `wiki-news-300d-1M.vec` ko bilkul `FastText.ipynb` wale folder me rakho** (side-by-side), warna `FileNotFoundError` aayega.
RAM kam hai (<=8 GB) to neeche cell me `limit=50000` hi rehne do — pura 1M load karne ki zaroorat nahi lab ke liye.

In [ ]:
import os
from gensim.models import KeyedVectors

model_path = "wiki-news-300d-1M.vec"   # .vec = plain .txt format hi hai

if not os.path.exists(model_path):
    fasttext_model = None
    print("Dataset nahi mila:", model_path)
    print("Upar wali markdown cell se download karke isi folder me rakho.")
    print("Tab tak Part 2 (Train your own) neeche bina dataset ke chalega.")
else:
    # limit=50000 -> sirf pehle 50k words load honge (fast + kam RAM). Pura chahiye to limit=None karo
    fasttext_model = KeyedVectors.load_word2vec_format(model_path, binary=False, limit=50000)
    print("Vocabulary size:", len(fasttext_model))
    print("Vector dimensions:", fasttext_model.vector_size)

In [ ]:
if fasttext_model is None:
    print('Pretrained model load nahi hua - is cell ko skip karo')
else:
    # Get the vector of a word
    
    word = "computer"
    
    vector = fasttext_model[word]
    
    print("Word:", word)
    print("Vector shape:", vector.shape)
    print("First 10 values:", vector[:10])


In [ ]:
if fasttext_model is None:
    print('Pretrained model load nahi hua - is cell ko skip karo')
else:
    #Find similar words
    
    similar_words = fasttext_model.most_similar(
        "computer",
        topn=5
    )
    
    for word, score in similar_words:
        print(word, round(score, 4))


In [ ]:
if fasttext_model is None:
    print('Pretrained model load nahi hua - is cell ko skip karo')
else:
    #Compare word similarities
    
    pairs = [
        ("king", "queen"),
        ("computer", "laptop"),
        ("computer", "banana")
    ]
    
    for word1, word2 in pairs:
        score = fasttext_model.similarity(word1, word2)
        print(word1, "-", word2, ":", round(score, 4))


In [ ]:
# Test a known word and an unseen word
# NOTE (fix): pehle yahan undefined `pretrained_ft.get_word_vector()` tha -> NameError aata tha.
# .vec/.txt se load hua KeyedVectors unseen (vocab ke bahar) word ka vector NAHI bana sakta,
# kyunki .vec file me subword (character n-gram) info save hi nahi hoti.
# Unseen-word wala magic sirf .bin pretrained model ya Part-2 me train kiye gaye model se hota hai (neeche dekho).
if fasttext_model is None:
    print('Pretrained model load nahi hua - skip')
else:
    print("playing in vocab:", "playing" in fasttext_model.key_to_index)
    print("Vector shape:", fasttext_model["playing"].shape)
    try:
        print(fasttext_model["playfullyxyz"].shape)
    except KeyError:
        print("'playfullyxyz' vocab me nahi hai -> .vec model iska vector nahi de sakta (yehi FastText .bin se alag hai)")

Both vectors should have 300 dimensions (pretrained). Unseen-word vector subword info se banta hai — iske liye `.bin` model ya khud train kiya model chahiye, sirf `.vec/.txt` file kaafi nahi.

## Train your own FastText model

In [ ]:
from gensim.models import FastText
from gensim.utils import simple_preprocess

sentences = [
    "machine learning is useful",
    "deep learning is a part of machine learning",
    "word embeddings represent words as vectors",
    "fasttext learns word representations from subwords",
    "word2vec learns useful word embeddings",
    "natural language processing uses word embeddings",
    "playing played player plays",
    "machine learning models learn patterns from data",
    "deep neural networks learn representations",
    "fasttext represents words using character ngrams"
]

tokenized_sentences = [
    simple_preprocess(sentence)
    for sentence in sentences
]

for sentence in tokenized_sentences[:3]:
    print(sentence)

In [ ]:
#Train the FastText model
ft_model = FastText(
    sentences=tokenized_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    min_n=3,
    max_n=6,
    epochs=100,
    seed=42
)

print("Vocabulary size:", len(ft_model.wv))
print("Vector dimensions:", ft_model.wv.vector_size)

In [ ]:
#Retrieve a trained word vector
word = "learning"

vector = ft_model.wv[word]

print("Word:", word)
print("Vector shape:", vector.shape)
print("First 10 values:", vector[:10])

In [ ]:
#Find similar words
print(
    ft_model.wv.most_similar(
        "learning",
        topn=5
    )
)

In [ ]:
known_word = "playing"
unseen_word = "playfullyxyz"

print("Known word in vocabulary:",
      known_word in ft_model.wv.key_to_index)

print("Unseen word in vocabulary:",
      unseen_word in ft_model.wv.key_to_index)

print("Known vector shape:",
      ft_model.wv[known_word].shape)

print("Unseen vector shape:",
      ft_model.wv[unseen_word].shape)

In [ ]:
#Save and reload your own model

ft_model.save("my_fasttext.model")

loaded_model = FastText.load("my_fasttext.model")

print(loaded_model.wv["learning"][:10])
print(loaded_model.wv["playfullyxyz"][:10])

---
### Chal gaya to kya output aana chahiye?
- Part 2 training ke baad: `Vocabulary size: ~35`, `Vector dimensions: 100`
- `ft_model.wv['learning']` -> shape `(100,)`
- `playfullyxyz` ka vector bhi Part-2 model de dega (shape `(100,)`) — yehi FastText ki khaasiyat hai
- Aakhir me `my_fasttext.model` file isi folder me save ho jayegi